# 17 — GA-based Counterfactual Search — DomesticDeclarations

Uses a genetic algorithm with constraint-preserving operators (Guidotti et al. 2024 adapted)
to generate counterfactual explanations for next-activity prediction.

Loads pre-mined constraints from 16a. No VAE needed (direct sequence mutation).

In [ ]:
import sys
import os
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
import torch

# --- Load dataset + prediction model ---
data_path = _current / 'encoded_data' / 'test_philipp' / 'domestic_declarations_all_5_test.pkl'
full_dataset = torch.load(data_path, weights_only=False)
dataset = full_dataset

sample = dataset[0]
n_cat = len(sample[0])
n_num = len(sample[1])
seq_len = sample[0][0].shape[0]
print(f'Dataset: {len(dataset)} sequences, {n_cat} cat, {n_num} num, seq_len={seq_len}')

from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

from src.interpretability.config.domestic_declarations_config import CONFIG
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(CONFIG.get_model_path()), dropout=0.0)
model.eval()
print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')

# --- TensorDecoder + activity vocabulary ---
from src.interpretability.utils.tensor_decoder import TensorDecoder

decoder = TensorDecoder(full_dataset)

ACTIVITY_FEATURE = 'Activity'
activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
max_idx = max(activity_idx_to_label.keys())
activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]
eos_idx = next(i for i, name in enumerate(activity_names) if name == 'EOS')

print(f'Activity vocabulary ({len(activity_names)}), EOS={eos_idx}')

In [ ]:
# Load pre-mined constraints
import pickle

constraints_pkl_path = _current / 'encoded_data' / 'domestic_declarations_constraints.pkl'
with open(constraints_pkl_path, 'rb') as f:
    constraints_data = pickle.load(f)

all_constraints = constraints_data['all']
print(f'Constraints: {len(all_constraints)} (from {constraints_pkl_path.name})')

In [ ]:
from src.interpretability.perturbation_methods import (
    GACounterfactual, GACounterfactualConfig, create_ga_counterfactual_for_model,
)

# Mutable categorical feature indices for DomesticDeclarations:
#   0 = Activity
#   1 = Resource
#   2 = Role (event-level)
MUTABLE_CAT_INDICES = [0, 1, 2]
CASE_LEVEL_CAT_INDICES = []  # No case-level features in DD

config = GACounterfactualConfig(
    population_size=200,
    n_generations=50,
    crossover_rate=0.8,
    mutation_rate=0.3,
    tournament_size=5,
    elite_size=5,
    mutable_cat_indices=MUTABLE_CAT_INDICES,
    case_level_cat_indices=CASE_LEVEL_CAT_INDICES,
    w_validity=5.0,
    w_proximity=1.0,
    w_sparsity=1.0,
    w_plausibility=1.0,
    w_conformance=2.0,
    top_k=5,
    diversity_threshold=0.3,
    activity_feature=ACTIVITY_FEATURE,
    verbose=True,
)

ga = create_ga_counterfactual_for_model(
    model=model,
    dataset=dataset,
    activity_names=activity_names,
    config=config,
    constraints=all_constraints,
    cat_feature_names=decoder.cat_features,
)
print(f'\nGA Counterfactual ready')

In [ ]:
import numpy as np
import pandas as pd

# === Scan dataset for candidate sequences ===
N_CANDIDATES = 50
scan_limit = min(500, len(dataset))

candidates = []
seen_cases = set()
for i in range(scan_limit):
    cat_t, num_t, case_id = dataset[i]
    act = cat_t[0]

    if case_id in seen_cases:
        continue
    seen_cases.add(case_id)

    trace_len = int((act != 0).sum().item())
    act_seq = [activity_names[a.item()] for a in act if a.item() != 0]

    candidates.append({
        'dataset_idx': i,
        'case_id': case_id,
        'trace_len': trace_len,
        'activities': ' -> '.join(act_seq),
    })

    if len(candidates) >= N_CANDIDATES:
        break

df_candidates = pd.DataFrame(candidates)
print(f'Found {len(candidates)} unique cases (scanned {scan_limit})')
print('Set SELECTED_ROW and PREFIX_LEN below to pick a case and prefix length.\n')
display(df_candidates)

In [ ]:
# ============================================
# SELECT A CASE AND PREFIX LENGTH
# ============================================
SELECTED_ROW = 0   # row index in the candidates table above
PREFIX_LEN = 3     # how many events to use as prefix (1 .. trace_len)

# --- Load and truncate to prefix ---
selected = df_candidates.iloc[SELECTED_ROW]
test_idx = selected['dataset_idx']
trace_len = selected['trace_len']
cat_tuple_full, num_tuple_full, case_id = dataset[test_idx]

prefix_len = max(1, min(PREFIX_LEN, trace_len))
if prefix_len != PREFIX_LEN:
    print(f'Note: PREFIX_LEN clamped to {prefix_len} (trace has {trace_len} events)')

# Build left-padded prefix tensors
pad_len = seq_len - prefix_len
cat_tuple = []
for c in cat_tuple_full:
    t = torch.zeros_like(c)
    src_start = seq_len - trace_len
    t[pad_len:] = c[src_start:src_start + prefix_len]
    cat_tuple.append(t)
cat_tuple = tuple(cat_tuple)

num_tuple = []
for n in num_tuple_full:
    t = torch.zeros_like(n)
    src_start = seq_len - trace_len
    t[pad_len:] = n[src_start:src_start + prefix_len]
    num_tuple.append(t)
num_tuple = tuple(num_tuple)

# Show prediction for this prefix
prefix_acts = [activity_names[cat_tuple[0][j].item()] for j in range(pad_len, seq_len)]
full_acts = [activity_names[a.item()] for a in cat_tuple_full[0] if a.item() != 0]

cat_in = [c.unsqueeze(0) for c in cat_tuple]
num_in = [n.unsqueeze(0) for n in num_tuple]
with torch.no_grad():
    preds = model((cat_in, num_in))[0]
    logits = preds[0][f'{ACTIVITY_FEATURE}_mean'][0]
    p = torch.softmax(logits, dim=-1)
    top_p, top_idx = p.max(dim=-1)

print(f'Case: {case_id}')
print(f'Full trace ({trace_len}): {" -> ".join(full_acts)}')
print(f'Prefix ({prefix_len}/{trace_len}):  {" -> ".join(prefix_acts)}')
print(f'Predicted next: {activity_names[top_idx.item()]} (p={top_p.item():.3f})')
print()
df_orig = decoder.decode_sequence(cat_tuple, num_tuple, case_id=case_id)
display(df_orig)

In [ ]:
# === Run GA counterfactual search ===
cat_tensors = [c.unsqueeze(0) for c in cat_tuple]
num_tensors = [n.unsqueeze(0) for n in num_tuple]

explanation = ga.explain(cat_tensors, num_tensors, target_class=None)
print(explanation)

In [ ]:
# === Display counterfactuals ===
if not explanation.counterfactuals:
    print('No counterfactuals found. Try increasing n_generations or population_size.')
else:
    # --- Original ---
    orig_acts_str = ' -> '.join(activity_names[a] for a in explanation.original_activity_sequence)
    print('=' * 100)
    print(f'ORIGINAL  —  Case: {case_id}')
    print(f'  {orig_acts_str}  ->  [{explanation.original_prediction_name}] (p={explanation.original_probability:.3f})')
    print('=' * 100)

    # Build original decoded values per mutable feature for diff
    orig_decoded = {}
    for ci in ga.mutable_cat_indices:
        feat_name = decoder.cat_features[ci]
        orig_tensor = cat_tuple[ci]
        orig_vals = [
            decoder.decode_categorical_value(feat_name, orig_tensor[j].item())
            for j in range(pad_len, seq_len)
        ]
        orig_decoded[ci] = orig_vals

    # --- Each counterfactual ---
    for i, cf in enumerate(explanation.counterfactuals):
        print(f'\n{"—" * 100}')
        # Activity sequence + new prediction
        cf_acts_str = ' -> '.join(activity_names[a] for a in cf.activity_sequence)
        print(f'CF #{i+1}:  {cf_acts_str}  ->  [{cf.counterfactual_prediction_name}] (p={cf.counterfactual_probability:.3f})')
        print(f'  fitness={cf.fitness:.4f}  prox={cf.proximity:.3f}  sparse={cf.sparsity}  conf={cf.conformance:.2f}  gen={cf.generation}')

        # Show only changed features
        cf_cat_squeezed = [t.squeeze(0) for t in cf.cat_sequence]
        changes = []
        for ci in ga.mutable_cat_indices:
            feat_name = decoder.cat_features[ci]
            cf_tensor = cf_cat_squeezed[ci]
            for pos_idx, pos in enumerate(range(pad_len, seq_len)):
                orig_val = orig_decoded[ci][pos_idx]
                cf_val = decoder.decode_categorical_value(feat_name, cf_tensor[pos].item())
                if orig_val != cf_val:
                    changes.append({
                        'position': pos_idx + 1,
                        'feature': feat_name,
                        'original': orig_val,
                        'counterfactual': cf_val,
                    })

        if changes:
            print(f'  Changes ({len(changes)}):')
            display(pd.DataFrame(changes))
        else:
            print('  No feature changes (prediction changed via numerical context)')